# leg_RAG — Generation (Module 4) walkthrough

A live, step-by-step run through `retrieval/hybrid_retrieve.py` ->
`generation/prompts.py` -> `generation/generate.py`: what actually gets
retrieved, how it's turned into a prompt, what the model returns, and the
two real bugs hit while building this (a Groq free-tier rate limit, and a
citation-bracket mismatch) with before/after evidence.

Run from the project root so `import config` resolves — this notebook finds
it automatically by walking upward, so it's safe to move.

In [1]:
import sys
import re
from pathlib import Path

_search = Path.cwd()
while not (_search / "config.py").exists():
    if _search.parent == _search:
        raise RuntimeError("Could not find project root (config.py) — open this notebook from within the leg_RAG project.")
    _search = _search.parent
PROJECT_ROOT = _search
sys.path.insert(0, str(PROJECT_ROOT))
import config
assert hasattr(config, "PROCESSED_DIR"), "Imported the wrong 'config' module — check for a stray pip package named 'config'."

from retrieval.hybrid_retrieve import hybrid_retrieve
from generation.prompts import build_messages, SYSTEM_PROMPT
from generation.generate import generate_answer

print("Provider:", config.GENERATION_PROVIDER)
print("Primary model:", config.GENERATION_MODEL_GROQ_PRIMARY)
print("Small-model comparison point:", config.GENERATION_MODEL_GROQ_SMALL)
print("Context chunks sent to generation:", config.GENERATION_MAX_CONTEXT_CHUNKS, "of", config.MAX_CANDIDATE_CHUNKS, "retrieved")

Provider: groq
Primary model: openai/gpt-oss-120b
Small-model comparison point: openai/gpt-oss-20b
Context chunks sent to generation: 12 of 20 retrieved


## 1. The system prompt

This is sent as the `system` message on every generation call — the rules
that require inline `[S#]` citation markers and forbid uncited claims.

In [2]:
print(SYSTEM_PROMPT)

You are a legal research assistant answering questions about federal case law, using only the source passages provided to you below.

Rules:
1. Every factual or legal claim in your answer must end with a citation marker referencing the specific source passage that supports it. Use plain ASCII square brackets exactly in this form: [S1] or [S2] — not full-width or curly brackets, not parentheses, nothing else. A sentence can carry more than one marker if it draws on multiple sources, e.g. "...excessive force. [S3][S8]"
2. Do not state anything as fact that isn't directly supported by one of the numbered source passages. If the passages don't contain enough information to answer the question, say so explicitly rather than filling the gap from general knowledge.
3. Do not treat any cited case as necessarily still good law — you are only summarizing what these passages say, not verifying whether the holdings have since been overruled or distinguished. A separate check handles that.
4. Write

## 2. Step one: retrieval

Runs the actual hybrid retrieval (vector search + graph expansion,
`retrieval/hybrid_retrieve.py`) for a real question — this is a live OpenAI
embedding call plus a local Chroma + graph lookup, same as module 3.

In [3]:
QUERY = "Can police use a taser on a suspect who is already handcuffed and restrained?"

chunks = hybrid_retrieve(QUERY)
n_vector = sum(1 for c in chunks if c["source"] == "vector")
n_graph = sum(1 for c in chunks if c["source"] == "graph")
print(f"{len(chunks)} candidates retrieved ({n_vector} vector, {n_graph} graph-expansion)\n")

for c in chunks:
    m = c["metadata"]
    print(f"[{c['source']:6s}] score={c['score']:.3f}  {m['case_name']}  ({m['date_filed']})")

20 candidates retrieved (8 vector, 12 graph-expansion)

[vector] score=0.611  Mattos v. Agarano  (2011-10-17)
[vector] score=0.598  Kyle Cardenas v. Josiah Saladen  (2023-03-02)
[vector] score=0.593  Heriberto Rodriguez v. County of Los Angeles  (2018-05-30)
[vector] score=0.592  Brooks v. City of Seattle  (2010-03-26)
[vector] score=0.591  Donald Gravelet-Blondin v. Sgt Jeff Shelton  (2013-09-06)
[vector] score=0.591  Isayeva v. Sacramento Sheriff's Department  (2017-10-02)
[vector] score=0.589  Jones v. Las Vegas Metropolitan Police Department  (2017-10-20)
[vector] score=0.588  Tucker v. Las Vegas Metropolitan Police Department  (2012-03-02)
[graph ] score=0.578  Johnson v. Bay Area Rapid Transit District  (2013-07-30)
[graph ] score=0.558  Daniel Barrera v. David Krause  (2023-02-17)
[graph ] score=0.535  Gerald Napouk v. Lvmpd  (2024-12-10)
[graph ] score=0.510  PAULETTE SMITH V. EDWARD AGDEPPA  (2022-12-30)
[graph ] score=0.509  Ian Tuuamalemalo v. Shahann Greene  (2019-12-24)
[g

## 3. Step two: building the prompt

`generation/prompts.py`'s `build_messages()` numbers the retrieved chunks
as `[S1]`, `[S2]`, ... and returns both the formatted block that goes into
the prompt, and a marker -> chunk metadata map used later to resolve a
citation back to its real source.

Generation only uses the first `GENERATION_MAX_CONTEXT_CHUNKS` of what
retrieval found — see the rate-limit bug in section 5 for why.

In [4]:
context_chunks = chunks[:config.GENERATION_MAX_CONTEXT_CHUNKS]
system_prompt, user_message, marker_map = build_messages(QUERY, context_chunks)

print(f"{len(marker_map)} sources numbered for this prompt.\n")
print("--- First ~800 characters of the actual user message sent to the model ---")
print(user_message[:800])

12 sources numbered for this prompt.

--- First ~800 characters of the actual user message sent to the model ---
Question: Can police use a taser on a suspect who is already handcuffed and restrained?

Source passages:

[S1] Mattos v. Agarano (2011-10-17)
 is no fun either). These methods are also distasteful to officers, who can deploy such close-range tactics only by stepping in harm’s way. The Taser is a safe alternative: It’s effective at a range of fifteen to thirty-five feet, so officers can use it without engag- ing in personal combat. And a study by six university depart- ments of emergency medicine found that 99.7 percent of those Tased by police suffer no injuries or, at most, mild ones. Wil- liam P. Bozeman et al., Safety and Injury Profile of Con- ducted Electrical Weapons Used by Law Enforcement Against Criminal Suspects, 53 Annals Emergency Med. 480, 484 (2009). The research division of


## 4. Step three: calling the model

A live call to Groq (`openai/gpt-oss-120b`). Note this model is a
*reasoning* model — it spends hidden tokens "thinking" before the visible
answer, which is part of why the token budget matters (section 5).

In [5]:
result = generate_answer(QUERY)

print(f"Provider: {result['provider']}\n")
print("Answer:")
print(result["answer"])
print()
print("Markers used:", result["markers_used"])
print("Markers retrieved but not cited:", result["markers_unused"])

Provider: groq

Answer:
Whether an officer may fire a Taser on a person who is already hand‑cuffed and otherwise restrained depends on how courts have interpreted the Fourth Amendment’s reasonableness standard in the handful of cases that address that exact situation.

* **The general rule is that a Taser is treated as “intermediate force” and may be justified only when the suspect is actively resisting or poses an ongoing threat.**  The Ninth Circuit has repeatedly held that a Taser is a Level‑2 tactic that “should be employed only against aggressive resistance.” [S4] In the Seventh Circuit, a suspect who was “already subdued” was the subject of a “tasing a suspect who was already subdued” discussion, and the court concluded that there was no “robust ‘consensus of cases of persuasive authority’” showing a Fourth‑Amendment violation, thereby granting the officers qualified immunity. [S2]

* **When the suspect is already hand‑cuffed and no longer resisting, several courts have found the

In [6]:
print("Sources, resolved back from marker to real case:\n")
for marker, src in result["sources"].items():
    used = "CITED" if marker in result["markers_used"] else "not cited"
    print(f"[{marker}] ({used:9s}) {src['case_name']}  ({src['date_filed']})  score={src['score']:.3f}")

Sources, resolved back from marker to real case:

[S1] (CITED    ) Mattos v. Agarano  (2011-10-17)  score=0.611
[S2] (CITED    ) Kyle Cardenas v. Josiah Saladen  (2023-03-02)  score=0.598
[S3] (CITED    ) Heriberto Rodriguez v. County of Los Angeles  (2018-05-30)  score=0.593
[S4] (CITED    ) Brooks v. City of Seattle  (2010-03-26)  score=0.592
[S5] (CITED    ) Donald Gravelet-Blondin v. Sgt Jeff Shelton  (2013-09-06)  score=0.591
[S6] (CITED    ) Isayeva v. Sacramento Sheriff's Department  (2017-10-02)  score=0.591
[S7] (CITED    ) Jones v. Las Vegas Metropolitan Police Department  (2017-10-20)  score=0.589
[S8] (CITED    ) Tucker v. Las Vegas Metropolitan Police Department  (2012-03-02)  score=0.588
[S9] (not cited) Johnson v. Bay Area Rapid Transit District  (2013-07-30)  score=0.578
[S10] (not cited) Daniel Barrera v. David Krause  (2023-02-17)  score=0.558
[S11] (not cited) Gerald Napouk v. Lvmpd  (2024-12-10)  score=0.535
[S12] (not cited) PAULETTE SMITH V. EDWARD AGDEPPA  (2022-

## 5. Two real bugs hit while building this

**Bug 1 — Groq free-tier rate limit.** The first attempt sent the full
`MAX_CANDIDATE_CHUNKS=20` chunks into the prompt. That request alone (system
prompt + 20 chunks + output token budget) came to ~8,268 tokens against
Groq's free-tier cap of 8,000 tokens/minute for `openai/gpt-oss-120b` — a
hard rejection, not a soft warning:

```
openai.APIStatusError: Error code: 413 - Request too large for model
`openai/gpt-oss-120b` ... tokens per minute (TPM): Limit 8000, Requested 8268
```

Fixed by capping generation's context at `GENERATION_MAX_CONTEXT_CHUNKS=12`
(retrieval itself still gathers up to 20 for other modules like
precedent-currency later — only generation's prompt is trimmed).

**Bug 2 — citation bracket mismatch.** The prompt asked for markers like
`[S1]`, and the model dutifully cited its sources — just using full-width
brackets (`【S3】`) instead of the ASCII brackets asked for. The marker-
extraction regex only looked for `[S1]`-style brackets, so it silently
reported *zero* citations used, even though the answer text clearly had
them. Demonstrated below on the literal first raw response we got.

In [7]:
bad_regex = r"\[S(\d+)\]"
good_regex = r"[\[\u3010{]S(\d+)[\]\u3011}]"

# The actual full-width-bracket citation style seen in the first raw Groq response:
raw_example = "...excessive force in tasing Keith\u3010S8\u3011 and further force after handcuffing."

print("Example text (full-width brackets, as the model actually returned it):")
print(f"  {raw_example}\n")
print("Old regex (ASCII-only) finds:", re.findall(bad_regex, raw_example))
print("Fixed regex (tolerates ASCII/full-width/curly) finds:", re.findall(good_regex, raw_example))

Example text (full-width brackets, as the model actually returned it):
  ...excessive force in tasing Keith【S8】 and further force after handcuffing.

Old regex (ASCII-only) finds: []
Fixed regex (tolerates ASCII/full-width/curly) finds: ['8']


Fixed two ways: tightened the prompt to spell out "plain ASCII square
brackets... not full-width or curly brackets, not parentheses" with an
inline example, *and* made the extraction regex tolerant of the variants
anyway — belt and suspenders, since prompt instructions aren't a hard
guarantee.

## 6. The hallucination this test run caught

On the run right after fixing both bugs above, the model's answer included
this bullet, with **no citation marker at all**:

> "The Tenth Circuit has also stressed that **'active resistance'** is a
> prerequisite for justified Taser deployment. In **Casey v. City of Fed.
> Heights**, the court held that using a Taser..."

`Casey v. City of Fed. Heights` is a real 10th Circuit case — but it is
**not among any of the 12 sources actually retrieved and provided** for
this query (check the source list in section 4 above). The model reached
into its own training knowledge instead of sticking to the provided
passages, despite the explicit system-prompt rule against it.

This isn't a bug to fix in the prompt — no amount of prompt engineering
fully prevents this. It's the exact failure mode module 5 (attribution +
entailment checking) exists to catch: decompose the answer into claims,
check each one against its cited source, and flag anything unsupported or
uncited. Worth keeping as a concrete example for the eventual write-up —
real evidence the problem this project targets actually happens, on the
very first real test.

## 7. Try your own question

Change the query below and re-run — full live round trip through
retrieval + generation.

In [8]:
MY_QUERY = "Does qualified immunity protect an officer who shoots a fleeing suspect who poses no threat?"

my_result = generate_answer(MY_QUERY)
print(f"Query: {my_result['query']}\n")
print(my_result["answer"])
print()
print("Markers used:", my_result["markers_used"])

Query: Does qualified immunity protect an officer who shoots a fleeing suspect who poses no threat?

Qualified immunity does **not** shield a police officer who shoots a fleeing suspect who does not pose a serious threat.  The Supreme Court has long held that “an officer may not shoot a fleeing suspect unless he poses a serious threat to the officer” — a rule that is “well‑established” and therefore “clearly established” law [S1].  Because qualified immunity “protects… officials… so long as their conduct ‘does not violate clearly established statutory or constitutional rights of which a reasonable person would have known’” [S3][S5][S7][S9][S10], an officer who violates the clearly‑established prohibition on shooting non‑threatening fleeing suspects would be acting contrary to a clearly established right and therefore would not be entitled to the immunity [S3][S5].  

In other words, when the facts show that the suspect was fleeing and posed no threat, the officer’s use of deadly force 

## Summary

- Full pipeline (retrieval -> prompt -> generation -> marker parsing) works
  end-to-end on a real question, with real citations resolved back to real
  cases.
- Hit and fixed two real bugs: a Groq free-tier token-rate limit, and a
  citation-bracket mismatch that was silently hiding real citations from
  the parser.
- Caught a live example of an uncited, unsupported claim slipping into an
  answer (`Casey v. City of Fed. Heights`) — direct motivation for module 5.

Next: module 5, attribution — decomposing the answer into atomic claims and
checking each one against its cited source passage.